## UrbanRide_13 — Demand Forecasting Model
**Type:** Regression — predicting a number, not a category  
**Label:** `trip_count` — number of trips per city per date  
**Source:** `urbanride.gold.trip_features` aggregated by city + date  
**Schedule:** Weekly · Sunday 04:00 AM (Job 4)

**Models trained:**
- Linear Regression — baseline
- Decision Tree Regressor — main model

**Metrics:** RMSE (primary) · MAE · R²

**Key difference vs Churn & Fraud:**
- Regression not Classification — label is a count not 0/1
- No class weights needed
- Row level = one row per city per date (~910 rows after aggregation)
- Feature engineering required — aggregate trip_features before training

**Experiment:** `/urbanride_demand_forecasting`  
**Registry:** `urbanride.default.urbanride_demand_model`  
**Output:** `urbanride.gold.demand_predictions`

In [0]:
# ── Run this cell first before anything else ─────────────────
import gc

stale_models = [
    'lr_model', 'dt_model', 'best_model',
    'BEST_MODEL', 'lr_pipeline', 'dt_pipeline'
]
for name in stale_models:
    try:
        del globals()[name]
        print(f'  Deleted: {name}')
    except KeyError:
        pass

gc.collect()
print('  Python memory cleared.')
print()

spark.sql('SELECT 1').show()
print('  Spark connection healthy.')

spark.sql('SHOW SCHEMAS IN urbanride').show()
print('  Catalog accessible.')

print()
print('Session ready. Safe to run notebook.')


  Python memory cleared.

+---+
|  1|
+---+
|  1|
+---+

  Spark connection healthy.
+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           landing|
|            silver|
+------------------+

  Catalog accessible.

Session ready. Safe to run notebook.


In [0]:
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import (
    col, count, avg, sum as spark_sum,
    round as spark_round, when, lit,
    current_timestamp, dayofweek, month,
    max as spark_max, min as spark_min
)
import pandas as pd
import gc
import tempfile
import os

CATALOG         = 'urbanride'
GOLD            = f'{CATALOG}.gold'
EXPERIMENT_NAME = '/urbanride_demand_forecasting'
MODEL_NAME      = 'urbanride.default.urbanride_demand_model'
SPLIT_DATE      = '2026-01-01'

# TMP_DIR — MLflow staging area during model save
TMP_DIR = '/Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp'
try:
    dbutils.fs.ls(TMP_DIR)
    print(f'TMP_DIR already exists: {TMP_DIR}')
except Exception:
    dbutils.fs.mkdirs(TMP_DIR)
    print(f'TMP_DIR created: {TMP_DIR}')

# Feature columns — all numeric after aggregation
# No categorical cols — city encoded via StringIndexer
NUMERIC_COLS = [
    'avg_fare_amount',
    'avg_distance_km',
    'avg_surge_multiplier',
    'avg_duration_minutes',
    'completed_trip_ratio',
    'cancelled_trip_ratio',
    'rainy_day_ratio',
    'peak_surge_ratio',
    'day_of_week',
    'month',
]

CATEGORICAL_COLS = ['city']
LABEL_COL        = 'trip_count'

mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Catalog          : {CATALOG}')
print(f'Source           : {GOLD}.trip_features')
print(f'Experiment       : {EXPERIMENT_NAME}')
print(f'Model name       : {MODEL_NAME}')
print(f'Split date       : {SPLIT_DATE}')
print(f'Numeric features : {len(NUMERIC_COLS)}')
print(f'Categorical      : {len(CATEGORICAL_COLS)}')
print('Config ready.')


2026/03/12 13:35:49 INFO mlflow.tracking.fluent: Experiment with name '/urbanride_demand_forecasting' does not exist. Creating a new experiment.


TMP_DIR already exists: /Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp
Catalog          : urbanride
Source           : urbanride.gold.trip_features
Experiment       : /urbanride_demand_forecasting
Model name       : urbanride.default.urbanride_demand_model
Split date       : 2026-01-01
Numeric features : 10
Categorical      : 1
Config ready.


In [0]:
print('Loading gold.trip_features...')

df_raw = spark.table(f'{GOLD}.trip_features')
total_trips = df_raw.count()
print(f'  Total trips : {total_trips:,}')
print()

# ── Aggregate to city + date level ────────────────────────────
# One row per city per date — this is what demand forecasting predicts
# From 19.6M trip rows → ~910 rows (5 cities × 182 days)
df_demand = df_raw.groupBy('city', 'trip_date').agg(

    # Label — what we predict
    count('trip_id').alias('trip_count'),

    # Fare & distance signals
    spark_round(avg('fare_amount'),        2).alias('avg_fare_amount'),
    spark_round(avg('distance_km'),        2).alias('avg_distance_km'),
    spark_round(avg('surge_multiplier'),   4).alias('avg_surge_multiplier'),
    spark_round(avg('trip_duration_minutes'), 2).alias('avg_duration_minutes'),

    # Trip quality ratios
    spark_round(
        avg(when(col('trip_status') == 'completed', 1).otherwise(0)), 4
    ).alias('completed_trip_ratio'),
    spark_round(
        avg(when(col('trip_status') == 'cancelled', 1).otherwise(0)), 4
    ).alias('cancelled_trip_ratio'),

    # Weather & surge signals
    spark_round(
        avg(when(col('weather') == 'Rain', 1).otherwise(0)), 4
    ).alias('rainy_day_ratio'),
    spark_round(
        avg(col('is_peak_surge').cast('int')), 4
    ).alias('peak_surge_ratio'),

).withColumn(
    # Calendar features — extracted from trip_date
    # day_of_week: 1=Sunday, 2=Monday ... 7=Saturday
    'day_of_week', dayofweek('trip_date')
).withColumn(
    'month', month('trip_date')
)

demand_count = df_demand.count()
print(f'Aggregated demand rows : {demand_count:,}')
print(f'Expected               : ~910 (5 cities × 182 days)')
print()
print('Sample demand data:')
df_demand.orderBy('city', 'trip_date').show(5)

print('Trip count stats by city:')
df_demand.groupBy('city').agg(
    spark_round(avg('trip_count'),  0).alias('avg_daily_trips'),
    spark_min('trip_count').alias('min_daily_trips'),
    spark_max('trip_count').alias('max_daily_trips'),
).orderBy('avg_daily_trips', ascending=False).show()


Loading gold.trip_features...
  Total trips : 21,359,745

Aggregated demand rows : 910
Expected               : ~910 (5 cities × 182 days)

Sample demand data:
+---------+----------+----------+---------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------+----------------+-----------+-----+
|     city| trip_date|trip_count|avg_fare_amount|avg_distance_km|avg_surge_multiplier|avg_duration_minutes|completed_trip_ratio|cancelled_trip_ratio|rainy_day_ratio|peak_surge_ratio|day_of_week|month|
+---------+----------+----------+---------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------+----------------+-----------+-----+
|Bengaluru|2025-09-01|     23786|         150.44|           6.97|              1.3745|               24.94|              0.9186|              0.0814|         0.5475|          0.2803|          2|    9|
|Bengaluru|2025-09-02|     23805|   

In [0]:
# Time-based split on trip_date
# Train : trips before Jan 1 2026  (~67%)
# Test  : trips on or after Jan 1 2026 (~33%)
# Same split date across all ML notebooks — consistent evaluation

df_train = df_demand.filter(col('trip_date') <  SPLIT_DATE)
df_test  = df_demand.filter(col('trip_date') >= SPLIT_DATE)

train_count = df_train.count()
test_count  = df_test.count()
total       = train_count + test_count

print(f'Split date : {SPLIT_DATE}')
print(f'Train rows : {train_count:,} ({train_count/total*100:.1f}%)')
print(f'Test rows  : {test_count:,}  ({test_count/total*100:.1f}%)')
print()

print('Avg daily trips in train vs test:')
print('Train:')
df_train.groupBy('city').agg(
    spark_round(avg('trip_count'), 0).alias('avg_daily_trips')
).orderBy('avg_daily_trips', ascending=False).show()

print('Test:')
df_test.groupBy('city').agg(
    spark_round(avg('trip_count'), 0).alias('avg_daily_trips')
).orderBy('avg_daily_trips', ascending=False).show()


Split date : 2026-01-01
Train rows : 610 (67.0%)
Test rows  : 300  (33.0%)

Avg daily trips in train vs test:
Train:
+---------+---------------+
|     city|avg_daily_trips|
+---------+---------------+
|    Delhi|        31738.0|
|   Mumbai|        27697.0|
|Bengaluru|        23452.0|
|Hyderabad|        19441.0|
|     Pune|        15167.0|
+---------+---------------+

Test:
+---------+---------------+
|     city|avg_daily_trips|
+---------+---------------+
|    Delhi|        32045.0|
|   Mumbai|        27676.0|
|Bengaluru|        23377.0|
|Hyderabad|        19211.0|
|     Pune|        14782.0|
+---------+---------------+



In [0]:
print('Building feature pipeline...')

# StringIndexer for city — 5 cities → 0.0 to 4.0
indexers = [
    StringIndexer(
        inputCol     = c,
        outputCol    = f'{c}_idx',
        handleInvalid= 'keep'
    )
    for c in CATEGORICAL_COLS
]

indexed_cat_cols = [f'{c}_idx' for c in CATEGORICAL_COLS]
ALL_INPUT_COLS   = NUMERIC_COLS + indexed_cat_cols

# VectorAssembler — combine all features into one vector
assembler = VectorAssembler(
    inputCols    = ALL_INPUT_COLS,
    outputCol    = 'features_raw',
    handleInvalid= 'skip'
)

# StandardScaler — needed for Linear Regression
# avg_fare_amount (~150) vs rainy_day_ratio (~0.3) — very different scales
scaler = StandardScaler(
    inputCol = 'features_raw',
    outputCol= 'features',
    withMean = True,
    withStd  = True
)

print(f'  Numeric features    : {len(NUMERIC_COLS)}')
print(f'  Categorical features: {len(CATEGORICAL_COLS)} → {len(indexed_cat_cols)} indexed')
print(f'  Total input cols    : {len(ALL_INPUT_COLS)}')
print('Feature pipeline ready.')


Building feature pipeline...
  Numeric features    : 10
  Categorical features: 1 → 1 indexed
  Total input cols    : 11
Feature pipeline ready.


In [0]:
print('Training Linear Regression (baseline)...')

# Regression evaluators — different from classification
# RMSE — primary: penalises large errors more (in trip units)
# MAE  — average absolute error (in trip units)
# R2   — how much variance is explained (0=nothing, 1=perfect)
rmse_evaluator = RegressionEvaluator(
    labelCol       = LABEL_COL,
    predictionCol  = 'prediction',
    metricName     = 'rmse'
)
mae_evaluator = RegressionEvaluator(
    labelCol       = LABEL_COL,
    predictionCol  = 'prediction',
    metricName     = 'mae'
)
r2_evaluator = RegressionEvaluator(
    labelCol       = LABEL_COL,
    predictionCol  = 'prediction',
    metricName     = 'r2'
)

lr = LinearRegression(
    featuresCol = 'features',
    labelCol    = LABEL_COL,
    maxIter     = 20,
    regParam    = 0.01,
)

lr_pipeline = Pipeline(stages=indexers + [assembler, scaler, lr])

with mlflow.start_run(run_name='linear_regression') as run:
    lr_run_id = run.info.run_id

    lr_model = lr_pipeline.fit(df_train)
    lr_preds = lr_model.transform(df_test)

    lr_rmse = rmse_evaluator.evaluate(lr_preds)
    lr_mae  = mae_evaluator.evaluate(lr_preds)
    lr_r2   = r2_evaluator.evaluate(lr_preds)

    mlflow.log_param('model_type', 'LinearRegression')
    mlflow.log_param('max_iter',   20)
    mlflow.log_param('reg_param',  0.01)
    mlflow.log_param('split_date', SPLIT_DATE)
    mlflow.log_metric('rmse', lr_rmse)
    mlflow.log_metric('mae',  lr_mae)
    mlflow.log_metric('r2',   lr_r2)

print(f'  LR RMSE : {lr_rmse:.2f} trips')
print(f'  LR MAE  : {lr_mae:.2f} trips')
print(f'  LR R²   : {lr_r2:.4f}')


Training Linear Regression (baseline)...
  LR RMSE : 9755.15 trips
  LR MAE  : 8732.78 trips
  LR R²   : -0.7505


In [0]:
print('Training Decision Tree Regressor...')

# Clear LR from cache before fitting DT
# Same pattern as Fraud notebook — avoids 1GB cache overflow
del lr_model
del lr_pipeline
gc.collect()
print('  LR cleared from cache.')

# Decision Tree does NOT need StandardScaler — scale invariant
assembler_dt = VectorAssembler(
    inputCols    = ALL_INPUT_COLS,
    outputCol    = 'features',
    handleInvalid= 'skip'
)

dt = DecisionTreeRegressor(
    featuresCol = 'features',
    labelCol    = LABEL_COL,
    maxDepth    = 10,
    seed        = 42,
)

dt_pipeline = Pipeline(stages=indexers + [assembler_dt, dt])

with mlflow.start_run(run_name='decision_tree_depth10') as run:
    dt_run_id = run.info.run_id

    dt_model = dt_pipeline.fit(df_train)
    dt_preds = dt_model.transform(df_test)

    dt_rmse = rmse_evaluator.evaluate(dt_preds)
    dt_mae  = mae_evaluator.evaluate(dt_preds)
    dt_r2   = r2_evaluator.evaluate(dt_preds)

    mlflow.log_param('model_type', 'DecisionTreeRegressor')
    mlflow.log_param('max_depth',  10)
    mlflow.log_param('split_date', SPLIT_DATE)
    mlflow.log_metric('rmse', dt_rmse)
    mlflow.log_metric('mae',  dt_mae)
    mlflow.log_metric('r2',   dt_r2)

print(f'  DT RMSE : {dt_rmse:.2f} trips')
print(f'  DT MAE  : {dt_mae:.2f} trips')
print(f'  DT R²   : {dt_r2:.4f}')


Training Decision Tree Regressor...
  LR cleared from cache.
  DT RMSE : 2837.65 trips
  DT MAE  : 1047.32 trips
  DT R²   : 0.8519


In [0]:
print('='*60)
print('DEMAND MODEL COMPARISON')
print('='*60)
print(f'{"Model":<35} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('-'*60)
print(f'{"Linear Regression":<35} {lr_rmse:>8.2f} {lr_mae:>8.2f} {lr_r2:>8.4f}')
print(f'{"Decision Tree (maxDepth=10)":<35} {dt_rmse:>8.2f} {dt_mae:>8.2f} {dt_r2:>8.4f}')
print('='*60)

# Pick best by RMSE — lower is better
if dt_rmse <= lr_rmse:
    BEST_MODEL      = dt_model
    BEST_RUN_ID     = dt_run_id
    BEST_RMSE       = dt_rmse
    BEST_MAE        = dt_mae
    BEST_R2         = dt_r2
    BEST_MODEL_TYPE = 'DecisionTreeRegressor_depth10'
    print('\nWinner: Decision Tree Regressor')
else:
    BEST_MODEL      = lr_model
    BEST_RUN_ID     = lr_run_id
    BEST_RMSE       = lr_rmse
    BEST_MAE        = lr_mae
    BEST_R2         = lr_r2
    BEST_MODEL_TYPE = 'LinearRegression'
    print('\nWinner: Linear Regression')

print(f'Best RMSE : {BEST_RMSE:.2f} trips')
print(f'Best MAE  : {BEST_MAE:.2f} trips')
print(f'Best R²   : {BEST_R2:.4f}')
print(f'\nCheck Experiments UI: {EXPERIMENT_NAME}')


DEMAND MODEL COMPARISON
Model                                   RMSE      MAE       R²
------------------------------------------------------------
Linear Regression                    9755.15  8732.78  -0.7505
Decision Tree (maxDepth=10)          2837.65  1047.32   0.8519

Winner: Decision Tree Regressor
Best RMSE : 2837.65 trips
Best MAE  : 1047.32 trips
Best R²   : 0.8519

Check Experiments UI: /urbanride_demand_forecasting


In [0]:
print('Registering best model to MLflow Model Registry...')

client = MlflowClient()

# Step 1 — Create model name in registry
try:
    client.create_registered_model(
        name        = MODEL_NAME,
        description = 'Forecasts daily trip demand per city for ZipCab ride-hailing platform.'
    )
    print(f'  Created model: {MODEL_NAME}')
except Exception:
    print(f'  Model already exists: {MODEL_NAME} — adding new version')

# Step 2 — Build model signature
sample_input = df_test.select(NUMERIC_COLS + CATEGORICAL_COLS).limit(100)
sample_pandas = sample_input.toPandas()

sample_preds  = BEST_MODEL.transform(sample_input)
sample_output = sample_preds.select('prediction').toPandas()

signature = infer_signature(
    model_input  = sample_pandas,
    model_output = sample_output
)
print('  Signature inferred.')
print(signature)

# Step 3 — Log model + register
with mlflow.start_run(run_name='register_demand_model') as reg_run:
    mlflow.log_param('model_type',    BEST_MODEL_TYPE)
    mlflow.log_param('best_rmse',     BEST_RMSE)
    mlflow.log_param('best_r2',       BEST_R2)
    mlflow.log_param('feature_count', len(ALL_INPUT_COLS))
    mlflow.log_metric('rmse', BEST_RMSE)
    mlflow.log_metric('mae',  BEST_MAE)
    mlflow.log_metric('r2',   BEST_R2)

    # Feature importance artifact (Decision Tree only)
    if 'DecisionTree' in BEST_MODEL_TYPE:
        dt_stage = BEST_MODEL.stages[-1]
        fi_df = pd.DataFrame({
            'feature':    ALL_INPUT_COLS,
            'importance': dt_stage.featureImportances.toArray()
        }).sort_values('importance', ascending=False)
        with tempfile.TemporaryDirectory() as tmp:
            fi_path = os.path.join(tmp, 'demand_feature_importance.csv')
            fi_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)
        print('  Feature importance logged.')

    mlflow.spark.log_model(
        spark_model   = BEST_MODEL,
        artifact_path = 'demand_model',
        signature     = signature,
        input_example = sample_pandas.head(5),
        dfs_tmpdir    = TMP_DIR
    )

    model_uri = f'runs:/{reg_run.info.run_id}/demand_model'
    mv = mlflow.register_model(model_uri, MODEL_NAME)

# Step 4 — Add version description
versions       = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max([int(v.version) for v in versions])

client.update_model_version(
    name        = MODEL_NAME,
    version     = str(latest_version),
    description = (
        f'{BEST_MODEL_TYPE}. RMSE={BEST_RMSE:.2f}. R²={BEST_R2:.4f}. '
        f'Trained on ZipCab 6 months data. '
        f'Split: trained before {SPLIT_DATE}. '
        f'Predicts daily trip count per city.'
    )
)
print(f'  Description added to {MODEL_NAME} v{latest_version}')

print()
print(f'  Model registered : {MODEL_NAME}')
print(f'  Version          : {latest_version}')
print(f'  Load URI         : models:/{MODEL_NAME}/{latest_version}')
print(f'  Check Models UI  : {MODEL_NAME}')


Registering best model to MLflow Model Registry...
  Model already exists: urbanride.default.urbanride_demand_model — adding new version


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


  Signature inferred.
inputs: 
  ['avg_fare_amount': double (required), 'avg_distance_km': double (required), 'avg_surge_multiplier': double (required), 'avg_duration_minutes': double (required), 'completed_trip_ratio': double (required), 'cancelled_trip_ratio': double (required), 'rainy_day_ratio': double (required), 'peak_surge_ratio': double (required), 'day_of_week': integer (required), 'month': integer (required), 'city': string (required)]
outputs: 
  ['prediction': double (required)]
params: 
  None

  Feature importance logged.


2026/03/12 13:38:37 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/12 13:38:40 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-508a3418-3cef-4d95-8794-91/tmpehw97w5j/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/12 13:38:41 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "avg_fare_amount",
      "avg_distance_km",
      "avg_surge_multiplier",
      "avg_duration_minutes",
      "completed_

  Description added to urbanride.default.urbanride_demand_model v1

  Model registered : urbanride.default.urbanride_demand_model
  Version          : 1
  Load URI         : models:/urbanride.default.urbanride_demand_model/1
  Check Models UI  : urbanride.default.urbanride_demand_model


In [0]:
print('Saving demand predictions to Delta table...')

# Score all city+date combinations
df_scored = BEST_MODEL.transform(df_demand)

df_predictions = df_scored.select(
    col('city'),
    col('trip_date'),
    col('trip_count').alias('actual_trip_count'),
    spark_round(col('prediction'), 0).alias('predicted_trip_count'),
    spark_round(
        (col('prediction') - col('trip_count')) / col('trip_count') * 100, 2
    ).alias('error_pct'),
    col('avg_fare_amount'),
    col('avg_surge_multiplier'),
    col('day_of_week'),
    col('month'),
).withColumn('_processed_at',  current_timestamp()) \
 .withColumn('_model_version', lit(str(latest_version))) \
 .withColumn('_model_name',    lit(MODEL_NAME))

df_predictions.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.demand_predictions')

pred_count = df_predictions.count()
print(f'  Predictions written : {pred_count:,}')
print(f'  Table               : {GOLD}.demand_predictions')


Saving demand predictions to Delta table...
  Predictions written : 910
  Table               : urbanride.gold.demand_predictions


In [0]:
print('=== DEMAND MODEL VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.demand_predictions')
total     = df_verify.count()

print(f'  Total predictions : {total:,}')
print(f'  Model             : {BEST_MODEL_TYPE}')
print(f'  Best RMSE         : {BEST_RMSE:.2f} trips')
print(f'  Best MAE          : {BEST_MAE:.2f} trips')
print(f'  Best R²           : {BEST_R2:.4f}')
print()

# Actual vs predicted by city
print('Actual vs predicted avg daily trips by city:')
df_verify.groupBy('city').agg(
    spark_round(avg('actual_trip_count'),    0).alias('avg_actual'),
    spark_round(avg('predicted_trip_count'), 0).alias('avg_predicted'),
    spark_round(avg('error_pct'),            2).alias('avg_error_pct'),
).orderBy('avg_actual', ascending=False).show()

# Actual vs predicted by day of week
print('Actual vs predicted by day of week (1=Sun, 7=Sat):')
df_verify.groupBy('day_of_week').agg(
    spark_round(avg('actual_trip_count'),    0).alias('avg_actual'),
    spark_round(avg('predicted_trip_count'), 0).alias('avg_predicted'),
).orderBy('day_of_week').show()

# Feature importance
if 'DecisionTree' in BEST_MODEL_TYPE:
    print('Top feature importances:')
    dt_stage = BEST_MODEL.stages[-1]
    fi = sorted(
        zip(ALL_INPUT_COLS, dt_stage.featureImportances.toArray()),
        key=lambda x: x[1],
        reverse=True
    )
    print(f'  {"Feature":<30} {"Importance":>10}')
    print(f'  {"-"*42}')
    for feat, imp in fi:
        print(f'  {feat:<30} {imp:>10.4f}')

print()
print('Sample predictions:')
df_verify.orderBy('city', 'trip_date').select(
    'city', 'trip_date', 'actual_trip_count',
    'predicted_trip_count', 'error_pct'
).show(10)

print()
print('='*55)
print('Demand model complete.')
print(f'Experiments UI : {EXPERIMENT_NAME}')
print(f'Models UI      : {MODEL_NAME}')
print(f'Predictions    : {GOLD}.demand_predictions')
print('Next           : UrbanRide_14 — Customer Segmentation')
print('='*55)


=== DEMAND MODEL VERIFICATION ===

  Total predictions : 910
  Model             : DecisionTreeRegressor_depth10
  Best RMSE         : 2837.65 trips
  Best MAE          : 1047.32 trips
  Best R²           : 0.8519

Actual vs predicted avg daily trips by city:
+---------+----------+-------------+-------------+
|     city|avg_actual|avg_predicted|avg_error_pct|
+---------+----------+-------------+-------------+
|    Delhi|   31839.0|      31603.0|         -0.6|
|   Mumbai|   27690.0|      27825.0|         0.66|
|Bengaluru|   23427.0|      23403.0|        -0.03|
|Hyderabad|   19365.0|      19495.0|         0.77|
|     Pune|   15040.0|      15222.0|          1.3|
+---------+----------+-------------+-------------+

Actual vs predicted by day of week (1=Sun, 7=Sat):
+-----------+----------+-------------+
|day_of_week|avg_actual|avg_predicted|
+-----------+----------+-------------+
|          1|   19045.0|      18771.0|
|          2|   24701.0|      24586.0|
|          3|   24335.0|      2446